# ASAP8 raw trace explorer

Small interactive notebook for walking through the extracted ASAP8 fluorescence trace.

- Uses `%matplotlib notebook` for zoom/pan interaction.
- `raw` is the extracted fluorescence with **no filtering**.
- Default `dff` is computed directly from raw fluorescence using one scalar F0 for the entire trace:
  `100 * (raw / F0 - 1)`.
- Edit `START_S` / `DURATION_S` or call `plot_window(...)` repeatedly to inspect candidate events.

In [ ]:
%matplotlib notebook

from pathlib import Path
import
import h5py
import numpy as np
import matplotlib.pyplot as plt

# ---------- Files ----------
SESSION_DIR = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+ASAP8\864742\864742_2026-08-20_08-36-59")  # change if notebook is not beside the data
DATA_DIR = 
TRACE_H5 = DATA_DIR / "dendriticVoltageTraces-260825-133020.h5"
SUMMARY_MAT = DATA_DIR / "dendriticVoltageSummary-260825-133020.mat"

# Sampling frequency for this SLAP2 extraction
FS = 10686.6669921875

# Scalar F0 used for the default, completely unfiltered dF/F conversion.
# 50 = median of the entire raw trace.
F0_PERCENTILE = 50

In [ ]:
# ---------- Load extracted raw fluorescence ----------

with h5py.File(TRACE_H5, "r") as f:
    trace_group = f["traces"]
    trial_keys = sorted(trace_group.keys())
    trial_traces = [np.asarray(trace_group[k])[0].astype(np.float64) for k in trial_keys]

raw = np.concatenate(trial_traces)
time_s = np.arange(raw.size) / FS

F0 = np.percentile(raw, F0_PERCENTILE)
dff = 100 * (raw / F0 - 1)

trial_lengths = np.array([x.size for x in trial_traces])
trial_edges_s = np.r_[0, np.cumsum(trial_lengths)] / FS

print(f"{len(trial_traces)} trials")
print(f"{raw.size:,} samples")
print(f"{raw.size / FS:.3f} s")
print(f"F0 ({F0_PERCENTILE}th percentile) = {F0:,.1f}")

In [ ]:
# ---------- Full-session overview ----------
# Downsample ONLY for display speed in this overview.
# The underlying raw and dff arrays remain full-resolution and unfiltered.

DISPLAY_DT_S = 0.02
step = max(1, int(round(DISPLAY_DT_S * FS)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_s[::step], raw[::step], lw=0.7)
ax.set(
    xlabel="Time (s)",
    ylabel="Raw fluorescence (a.u.)",
    title="Full-session raw fluorescence",
)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

In [ ]:
# ---------- Interactive window browser ----------

def plot_window(start_s=0, duration_s=2, signal="raw", mark_trials=True, figsize=(10, 4)):
    """
    Plot a full-resolution, unfiltered segment.

    signal:
        "raw" -> extracted raw fluorescence
        "dff" -> 100 * (raw / scalar F0 - 1), with no temporal filtering
    """
    signal = signal.lower()
    if signal not in {"raw", "dff"}:
        raise ValueError("signal must be 'raw' or 'dff'")

    y = raw if signal == "raw" else dff
    ylabel = "Raw fluorescence (a.u.)" if signal == "raw" else "ΔF/F (%)"

    start = max(0, int(round(start_s * FS)))
    stop = min(y.size, int(round((start_s + duration_s) * FS)))

    if stop <= start:
        raise ValueError("Requested window is outside the recording.")

    t = time_s[start:stop]

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(t, y[start:stop], lw=0.8)

    if mark_trials:
        edges = trial_edges_s[(trial_edges_s > t[0]) & (trial_edges_s < t[-1])]
        for edge in edges:
            ax.axvline(edge, ls="--", lw=0.7, alpha=0.35)

    ax.set(
        xlabel="Time (s)",
        ylabel=ylabel,
        title=f"{signal.upper()} | {t[0]:.3f}–{t[-1]:.3f} s",
    )
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    return fig, ax

# Edit these and re-run:
START_S = 165.0
DURATION_S = 1.0
SIGNAL = "dff"  # "raw" or "dff"

plot_window(START_S, DURATION_S, SIGNAL)

In [ ]:
# ---------- Convenience: jump directly to a candidate time ----------

def plot_around(center_s, window_ms=120, signal="dff"):
    half_s = window_ms / 2000
    return plot_window(
        start_s=center_s - half_s,
        duration_s=2 * half_s,
        signal=signal,
        mark_trials=False,
        figsize=(8, 3.5),
    )

# Candidate times from the quick-look pass:
candidate_times_s = [165.21, 172.20, 173.36, 207.98, 239.73, 255.76]

plot_around(candidate_times_s[0], window_ms=120, signal="dff")

In [ ]:
# ---------- Optional: raw and dF/F over the same window ----------

def plot_raw_and_dff(start_s=165, duration_s=1):
    start = max(0, int(round(start_s * FS)))
    stop = min(raw.size, int(round((start_s + duration_s) * FS)))
    t = time_s[start:stop]

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(t, raw[start:stop], lw=0.8)
    axes[0].set_ylabel("Raw fluorescence")
    axes[0].spines[["top", "right"]].set_visible(False)

    axes[1].plot(t, dff[start:stop], lw=0.8)
    axes[1].set(xlabel="Time (s)", ylabel="ΔF/F (%)")
    axes[1].spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    return fig, axes

plot_raw_and_dff(165.0, 1.0)

### Notes

The default `dff` above is intentionally simple and **does not correct bleaching**. It only rescales the raw fluorescence by one scalar F0, so the waveform itself is not temporally filtered.

If you later want bleaching-corrected dF/F, add a slowly varying F0 estimate separately and compare it against `raw`.